# 12. Calling it yourself

Run the cells from top to bottom. Select the `mano` kernel. The working directory may be `MANO` or `MANO/tutorial`.

`require_model()` reads `mano_v1_2/models/MANO_RIGHT.pkl`. Edit any cell below and run it again.

## The flat template


In [ ]:
import sys
from pathlib import Path

import numpy as np

MANO = Path.cwd().resolve()
if not (MANO / "mano_skeleton.py").exists():
    MANO = MANO.parent
if str(MANO) not in sys.path:
    sys.path.insert(0, str(MANO))

from official_mano import require_model

model = require_model()
flat = model.forward(flat_hand=True)

vertices = flat["vertices"]
joints = flat["joints"]
index_tip = flat["tips"]["index"]
faces = model.faces

print("vertices", vertices.shape)
print("joints  ", joints.shape)
print("faces   ", faces.shape)
print("tips    ", {name: np.round(point, 4) for name, point in flat["tips"].items()})


vertices (778, 3)
joints   (16, 3)
faces    (1538, 3)
tips     {'thumb': array([ 0.0037, -0.0164,  0.0941]), 'index': array([-0.0727,  0.0051,  0.0285]), 'middle': array([-0.0788,  0.0094, -0.0108]), 'ring': array([-0.0623,  0.0067, -0.0406]), 'pinky': array([-0.0231, -0.0026, -0.0702])}


With no arguments, global rotation, finger pose, shape, and translation are all 0. The default of `flat_hand` is `True`.

The keys of `tips` are `thumb`, `index`, `middle`, `ring`, and `pinky`.

## Flex the index finger, and change the body shape


In [ ]:
hand_pose = np.zeros(45)
hand_pose[2] = np.deg2rad(40)  # index MCP, about +Z, toward the palm
hand_pose[5] = np.deg2rad(65)  # index PIP
hand_pose[8] = np.deg2rad(45)  # index DIP

betas = np.zeros(10)
betas[0] = -2.0

out = model.forward(
    global_orient=np.zeros(3),
    hand_pose=hand_pose,
    betas=betas,
    transl=np.zeros(3),
    flat_hand=True,
)
print("index tip", np.round(out["tips"]["index"], 4))
print("wrist to middle tip, mm", round(float(np.linalg.norm(out["tips"]["middle"] - out["joints"][0]) * 1000), 1))


index tip [ 0.0094 -0.0543  0.0321]
wrist to middle tip, mm 191.5


Global rotation is another 3 numbers. It is not written into `hand_pose`, and it does not enter the 135 pose-blend coefficients.

## Translation, and rotating the whole hand


In [ ]:
reference = model.forward(flat_hand=True)
shifted = model.forward(transl=np.array([0.1, 0.0, 0.2]), flat_hand=True)
turned = model.forward(global_orient=np.array([0.3, -0.2, 0.4]), flat_hand=True)

print("wrist motion from translation alone", np.round(shifted["joints"][0] - reference["joints"][0], 4))
before = np.linalg.norm(reference["joints"][1] - reference["joints"][4])
after = np.linalg.norm(turned["joints"][1] - turned["joints"][4])
print("index-MCP to middle-MCP distance, rotation only", before, after)
assert np.allclose(shifted["joints"] - reference["joints"], np.array([0.1, 0.0, 0.2]))
assert np.isclose(before, after)


wrist motion from translation alone [0.1 0.  0.2]
index-MCP to middle-MCP distance, rotation only 0.025199516349807074 0.025199516349807074


A pure `transl` adds the same vector to all 16 joints and to every vertex. A pure global rotation keeps distances between joints. The wrist is not at the origin, so a rotation also changes the wrist coordinate.

## PCA

The mean pose is 6 coefficients of 0. Here `hand_pose` holds coefficients, not axis-angles.


In [ ]:
mean_pose = model.forward(hand_pose=np.zeros(6), flat_hand=False, pose_is_pca=True)
print("first 9 entries of full_pose (3 global, then the first index axis-angles)")
print(np.round(mean_pose["full_pose"][:9], 4))


first 9 entries of full_pose (3 global, then the first index axis-angles)
[ 0.      0.      0.      0.1117 -0.0429  0.4164  0.1088  0.066   0.7562]


## Return values

| Key | Shape | Meaning |
|---|---|---|
| `vertices` | (778, 3) | The mesh after skinning and translation |
| `joints` | (16, 3) | Forward-kinematics joints, with translation added |
| `tips` | 5 vectors of length 3 | Fingertip vertices on the current mesh |
| `full_pose` | (48,) | `global_orient` followed by the final 45 finger axis-angles |
| `joints_rest` | (16, 3) | Joints after the shape blend and before rotation |

Draw with `vertices` and `faces`. Align keypoints with `joints` and `tips`.

## Exercise

Flex only the ring PIP by about 50 degrees, toward the palm of this right hand, and compare how far the ring tip and the index tip move. The ring PIP is joint 11. Its `hand_pose` slice is `[30:33]`, and rz is index 32.


In [ ]:
hand_pose = np.zeros(45)
hand_pose[32] = np.deg2rad(50)
bent = model.forward(hand_pose=hand_pose, flat_hand=True)
flat = model.forward(flat_hand=True)
print("ring tip moved mm", round(float(np.linalg.norm(bent["tips"]["ring"] - flat["tips"]["ring"]) * 1000), 1))
print("index tip moved mm", round(float(np.linalg.norm(bent["tips"]["index"] - flat["tips"]["index"]) * 1000), 3))
assert np.linalg.norm(bent["tips"]["ring"] - flat["tips"]["ring"]) > 0.01
assert np.linalg.norm(bent["tips"]["index"] - flat["tips"]["index"]) < 1e-4


ring tip moved mm 40.5
index tip moved mm 0.001
